<a href="https://colab.research.google.com/github/ycc90123/omnizart-colab-fixed/blob/main/omnizart_colab_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Omniscient Mozart

This is a colab for demonstrating the python package `omnizart` developed by [MCTLab](https://sites.google.com/view/mctl/home).

Github repository can be found in [Music-and-Culture-Technology-Lab/omnizart](https://github.com/Music-and-Culture-Technology-Lab/omnizart).

Official documentation page can be found in [omnizart-doc](https://music-and-culture-technology-lab.github.io/omnizart-doc/)

In [ ]:
# @title 🛠️ Environment Setup
import os
import sys

# ==========================================
# 1. 安裝 Miniforge
# ==========================================
if not os.path.exists('/root/miniforge3'):
    print("⏳ 正在安裝 Miniforge...")
    !wget -qO Miniforge3.sh "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh"
    !bash Miniforge3.sh -b -p /root/miniforge3
    !rm Miniforge3.sh
else:
    print("👌 Miniforge 已安裝。")

# ==========================================
# 2. 建立 Python 3.8 環境
# ==========================================
print("⏳ 正在配置 Python 3.8 環境...")
conda_path = "/root/miniforge3/bin/conda"
!{conda_path} create -n omnizart_env python=3.8 numpy=1.19.5 -y

# ==========================================
# 3. 安裝系統音訊工具
# ==========================================
print("⏳ 正在安裝系統依賴...")
!apt-get update -qq
!apt-get install -y libsndfile1 fluidsynth ffmpeg > /dev/null

# ==========================================
# 4. 依照順序安裝 Python 套件
# ==========================================
print("⏳ 正在安裝套件 (這是一場精密手術)...")
env_path = "/root/miniforge3/envs/omnizart_env"
pip_path = f"{env_path}/bin/pip"

# 4-1. 基礎升級
!{pip_path} install --upgrade pip==21.3.1 setuptools wheel

# 4-2. 編譯工具
print("👉 [1/6] 安裝 Cython...")
!{pip_path} install Cython numpy==1.19.5

# 4-3. 核心衝突 (Librosa)
print("👉 [2/6] 安裝 Librosa 堆疊...")
!{pip_path} install librosa==0.8.0 numba==0.53.0 llvmlite==0.36.0

# 4-4. 大型框架
print("👉 [3/6] 安裝 Spleeter 與 TensorFlow...")
!{pip_path} install spleeter==2.3.0
!{pip_path} install tensorflow==2.5.0

# 4-5. Madmom
print("👉 [4/6] 安裝 Madmom...")
!{pip_path} install madmom

# 4-6. 補丁 (關鍵修正點！)
print("👉 [5/6] 安裝所有相依套件...")
# 這裡使用雙引號包住整串字串，避免 < 被誤判
# 並且將 tqdm<5.0.0 用引號特別包起來
patch_packages = (
    "click==7.1.2 pyyaml==5.4.1 urllib3==1.26.4 pillow==8.4.0 "
    "jsonschema==3.2.0 mir_eval==0.6 pyfluidsynth==1.3.0 vamp==1.1.0 "
    "pretty_midi yt-dlp \"tqdm<5.0.0\""
)
!{pip_path} install {patch_packages}

# 4-7. Omnizart 本體
print("👉 [6/6] 安裝 Omnizart...")
!{pip_path} install git+https://github.com/Music-and-Culture-Technology-Lab/omnizart.git --no-deps

# ==========================================
# 5. 指令捷徑 & 下載模型
# ==========================================
print("⏳ 製作捷徑與下載模型...")
omnizart_real_path = f"{env_path}/bin/omnizart"
wrapper_script = f"""#!/bin/bash
source /root/miniforge3/etc/profile.d/conda.sh
conda activate omnizart_env
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:/root/miniforge3/envs/omnizart_env/lib
exec {omnizart_real_path} "$@"
"""
with open("/usr/local/bin/omnizart", "w") as f:
    f.write(wrapper_script)
!chmod +x /usr/local/bin/omnizart

!omnizart download-checkpoints

print("\n✅ 環境全數建置完成！")

# Choose an Audio

Either upload your own music file, or choose from YouTube.


In [ ]:
# @title Upload MP3 File
import os
from google.colab import files
from IPython import display as dsp

# 上傳檔案
ff = files.upload()

# 取得檔名 (去除副檔名)
uploaded_audio = list(ff.keys())[0]
file_base_name = os.path.splitext(uploaded_audio)[0]

# 轉檔為 WAV (確保 ffmpeg 存在)
print("Converting to WAV...")
!ffmpeg -y -i "{uploaded_audio}" "{file_base_name}.wav" &>/dev/null

# 設定變數給後面用
target_file = file_base_name + ".wav"

print(f"✅ 檔案準備完成: {target_file}")
# 播放預覽
dsp.Audio(target_file) if os.path.exists(target_file) else None

In [ ]:
# @title Choose from YouTube
import os
from google.colab import files
from IPython import display as dsp

# 確保 yt-dlp 存在 (這行是新加的保險)
!pip install -q yt-dlp

url = input("Enter your YouTube link: ")

# 嘗試顯示影片預覽
try:
  if "watch?v=" in url:
    id = url.split("watch?v=")[1].split("&")[0]
    vid = dsp.YouTubeVideo(id)
    dsp.display(vid)
  elif "youtu.be" in url:
    id = url.split("/")[-1]
    vid = dsp.YouTubeVideo(id)
    dsp.display(vid)
except Exception:
  pass

print("Downloading...")

# 下載並轉檔
!yt-dlp -x --audio-format wav --no-playlist -o "%(title)s.%(ext)s" "{url}"
!yt-dlp --get-filename --no-playlist -o "%(title)s.wav" "{url}" > tmp

# 讀取下載後的檔名
with open("tmp", "r") as f:
    target_file = f.readline().strip()

print(f"✅ Finished: {target_file}")

# Transcribe the Audio

There are several modes you can choose.
* `music-piano`: transcribe piano solo clips.
* `music-assemble`: transcribe classical assemble pieces.
* `chord`: transcribe chord progressions.
* `drum`: transcribe drum percussion in the audio.
* `vocal`: transcribe note-level vocal notes.
* `vocal-contour`: transcribe frame-level vocal pitch contour.
* `beat`: transcribe beat and down beat positions on symbolic domain *(see note 1)*.

## Notes
1. The beat module only supports MIDI inputs, and thus you have to upload the MIDI file through the **Upload MP3 File** block.

In [ ]:
#@title 🎹 Transcribe & Play
import os
import sys

# ==========================================
# 1. Auto-fix dependencies
# ==========================================
# Ensure Colab has the necessary tools for synthesis
try:
    from pretty_midi import PrettyMIDI
except ImportError:
    print("⏳ Installing synthesis tools (pretty_midi, pyfluidsynth)...")
    !pip install -q pretty_midi pyfluidsynth librosa numpy scipy
    from pretty_midi import PrettyMIDI

import librosa
import numpy as np
import scipy.io.wavfile
from IPython import display as dsp

# ==========================================
# 2. Setup parameters & Fetch file
# ==========================================
mode = "drum" #@param ["music-piano", "music-piano-v2", "music-assemble", "chord", "drum", "vocal", "vocal-contour", "beat"]

# Automatically detect the latest WAV file in the directory
# Avoids variable name conflicts (uploaded_audio vs target_file)
wavs = [f for f in os.listdir('.') if f.endswith('.wav') and "_synth" not in f]
if not wavs:
    print("❌ Error: No .wav file found! Please upload or download a file first.")
    sys.exit()

# Get the newest file
target_file = max(wavs, key=os.path.getctime)
uploaded_audio = os.path.splitext(target_file)[0] # Get filename without extension for compatibility
print(f"⚠️ Processing file: {target_file}")

# ==========================================
# 3. Build and Execute Omnizart Command
# ==========================================
model = ""
if mode.startswith("music"):
    mode_list = mode.split("-")
    run_mode = mode_list[0] # Actual command is 'music'
    model = "-".join(mode_list[1:])
else:
    run_mode = mode

# Map model path parameters
model_path_map = {
    "piano": "Piano",
    "piano-v2": "PianoV2",
    "assemble": "Stream",
    "pop-song": "Pop",
    "": None
}
model_path = model_path_map.get(model, None)

print(f"🚀 Executing Omnizart ({mode})...")

# Assemble command (use double quotes to handle spaces/non-ASCII characters in filenames)
if model_path:
    !omnizart "$run_mode" transcribe "$target_file" --model-path "$model_path"
else:
    !omnizart "$run_mode" transcribe "$target_file"

# ==========================================
# 4. Synthesize MIDI & Play
# ==========================================
# Predict output filename (Omnizart naming convention)
possible_outputs = [
    f"{uploaded_audio}.mid",           # General music mode
    f"{uploaded_audio}_{mode}.mid",    # drum, vocal modes usually add suffix
    f"{uploaded_audio}_drum.mid",
    f"{uploaded_audio}_chord.mid",
    f"{uploaded_audio}_vocal.mid",
    f"{uploaded_audio}_trans.mid"      # General backup
]

midi_file = None
for f in possible_outputs:
    if os.path.exists(f):
        midi_file = f
        break

# Special case: vocal-contour outputs wav, not mid
if mode == "vocal-contour":
    midi_file = f"{uploaded_audio}_trans.wav"

if not midi_file or not os.path.exists(midi_file):
    print("❌ Transcription failed. Output file not found.")
else:
    print(f"✅ Transcription successful! Output file: {midi_file}")

    # Download SoundFont (if missing)
    SF2_FILE = "general_soundfont.sf2"
    if not os.path.exists(SF2_FILE):
        print("📥 Downloading SoundFont...")
        !curl -L "https://ftp.osuosl.org/pub/musescore/soundfont/MuseScore_General/MuseScore_General.sf2" -o {SF2_FILE}

    synth_name = f"{uploaded_audio}_synth.wav"

    try:
        if mode == "vocal-contour":
            # If it's already wav, copy it directly
            import shutil
            shutil.copy(midi_file, synth_name)
        else:
            print("🎹 Synthesizing preview audio...")
            midi_data = PrettyMIDI(midi_file)
            raw_wav = midi_data.fluidsynth(fs=44100, sf2_path=SF2_FILE)

            # Normalize to prevent clipping
            max_val = np.abs(raw_wav).max()
            if max_val > 0: raw_wav = raw_wav / max_val * 0.9

            # Write using scipy (more stable than manual functions)
            scipy.io.wavfile.write(synth_name, 44100, (raw_wav * 32767).astype(np.int16))

        # Convert to MP3 for browser playback
        out_name = f"{uploaded_audio}_synth.mp3"
        !ffmpeg -y -i "$synth_name" "$out_name" &> /dev/null

        print(f"🎉 Done!")
        dsp.display(dsp.Audio(out_name))

        # Auto-download removed
        print(f"📂 Your MIDI file is located at: {midi_file}")
        print("To download, please click the 'Folder' icon on the left.")

    except Exception as e:
        print(f"⚠️ Error during synthesis/playback: {e}")
        print("However, the MIDI file is saved. You can download it manually.")

# Download the Transribed MIDI/MP3

In [ ]:
# @title 📥 Download Results (MIDI & MP3)
import os
from google.colab import files

# 1. Automatically search for result files in the directory
print("🔍 Searching for downloadable files...")

# Find all MIDI files
midi_files = [f for f in os.listdir('.') if f.endswith('.mid')]

# Find all synthesized MP3 files
mp3_files = [f for f in os.listdir('.') if f.endswith('_synth.mp3')]

# 2. Download MIDI files
if midi_files:
    print(f"📂 Found {len(midi_files)} MIDI file(s), preparing download...")
    for midi in midi_files:
        print(f"  ⬇️ Downloading: {midi}")
        files.download(midi)
else:
    print("⚠️ No MIDI files found.")

# 3. Download MP3 files
if mp3_files:
    print(f"📂 Found {len(mp3_files)} synthesized audio file(s), preparing download...")
    for mp3 in mp3_files:
        print(f"  ⬇️ Downloading: {mp3}")
        files.download(mp3)
else:
    print("ℹ️ No synthesized MP3 files found (this is normal if you skipped synthesis).")